# Prediction uncertainty: entropy and top-100 mass

For each test disease the model predicts the drug at the masked position of `[disease, relation, MASK, ...]`.
The softmax is taken over all drugs only (the valid answers), and per disease we compute

- **entropy** H = -sum p log p, normalised to 0-1 (H / log(number of drugs)) and as **effective number of candidates** exp(H);
- **top-100 mass**: summed probability of the 100 highest-ranked drugs;
- **Recall@100** of the held-out test drugs of that disease (as in node_level_eval).

If the uncertainty is informative, diseases with a high top-100 mass / low entropy have a higher Recall@100.

In [ ]:
import os
import sys
if os.path.basename(os.getcwd()) == "revision_scripts":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import stats

from netmedgpt.model import TransformerModel
from netmedgpt.prepare_txgnn_splits import prepare_txgnn_data

CHECKPOINT = "data/model_checkpoints/saved_models_evaluation/netmedgpt_randomLinkSplit_seed1.pt"
SPLIT_DIR = "data/splits/random_1"
RELATIONS = ["indication", "contraindication", "off-label use"]   # all disease -> drug
K = 100
GPU = 0
OUT_DIR = "data/new_analysis/prediction_uncertainty"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ── KG, model (same setup as other/evaluate_checkpoints.py) ───────────────────
all_param = json.load(open("data/parameters.json"))
nodes = pd.read_csv("data/KG/nodes.csv")
edge = pd.read_csv("data/KG/edges.csv")
feat = torch.load("data/KG/embeddings.pt")

mask_token = int(edge["z_index"].max()) + 1
relation_type = list(edge["relation"].unique())
entity = list(nodes["node_type"].unique()) + relation_type + ["mask"]
relation_mask_index = pd.concat([edge[["relation", "z_index"]].drop_duplicates(),
                                 pd.DataFrame([["mask", mask_token]], columns=["relation", "z_index"])], ignore_index=True)
rel_token = {r: int(edge.loc[edge["relation"] == r, "z_index"].iloc[0]) for r in RELATIONS}
drug_ids = torch.tensor(nodes.loc[nodes["node_type"] == "drug", "node_index"].unique(), dtype=torch.long)

device = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")
hp = all_param["best_hyperparam"]
seq_len = all_param["node2vec"]["walk_length"] * 2 - 1
ckpt = torch.load(CHECKPOINT, map_location=device)
state_dict = ckpt["model_state"] if "model_state" in ckpt else ckpt     # two saved formats
model = TransformerModel(mask_token + 1, hp["hidden_channels"], hp["nhead"], hp["N_encoder_layers"], seq_len,
                         device=device, feat=feat, nodes=nodes, entity=entity,
                         relation_mask_index=relation_mask_index, pos_emb="fixed").to(device)
model.load_state_dict(state_dict)
model.eval()

# held-out test edges, oriented disease -> drug
splits = [pd.read_csv(os.path.join(SPLIT_DIR, f), low_memory=False) for f in ["train.csv", "valid.csv", "test.csv"]]
_, _, test = prepare_txgnn_data(nodes, *splits, {r: "disease|drug" for r in RELATIONS})

In [ ]:
# ── Entropy, top-100 mass and Recall@100 per test disease ─────────────────────
rows = []
for rel in RELATIONS:
    t = test[test["relation"] == rel]
    diseases = torch.tensor(t["x_index"].unique(), dtype=torch.long)
    sentences = torch.full((len(diseases), seq_len), mask_token, dtype=torch.long)
    sentences[:, 0] = diseases
    sentences[:, 1] = rel_token[rel]
    with torch.no_grad():
        logits = model(sentences.to(device))[:, 2, :][:, drug_ids.to(device)]   # drugs only
    p = F.softmax(logits.float(), dim=-1)
    H = -(p * torch.log(p + 1e-12)).sum(-1)
    top_p, top_i = p.topk(K, dim=-1)
    top_drugs = drug_ids[top_i.cpu()]

    for i, d in enumerate(diseases.tolist()):
        true = set(t.loc[t["x_index"] == d, "y_index"])
        rows.append({"relation": rel, "disease": d, "n_test_drugs": len(true),
                     "entropy": H[i].item(),
                     "entropy_norm": H[i].item() / math.log(len(drug_ids)),
                     "n_effective": math.exp(H[i].item()),
                     "top100_mass": top_p[i].sum().item(),
                     "recall@100": len(true & set(top_drugs[i].tolist())) / len(true)})

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, "per_disease.csv"), index=False)
print(f"{len(res)} test diseases")

In [ ]:
# ── Is the uncertainty informative? ───────────────────────────────────────────
summary = []
for rel, g in res.groupby("relation", sort=False):
    rho_mass, p_mass = stats.spearmanr(g["top100_mass"], g["recall@100"])
    rho_h, p_h = stats.spearmanr(g["entropy"], g["recall@100"])
    q = pd.qcut(g["top100_mass"], 4, labels=["Q1 (least confident)", "Q2", "Q3", "Q4 (most confident)"])
    summary.append({"relation": rel, "diseases": len(g),
                    "median top100_mass": g["top100_mass"].median(),
                    "median n_effective": g["n_effective"].median(),
                    "Spearman(top100_mass, recall)": rho_mass, "p (mass)": p_mass,
                    "Spearman(entropy, recall)": rho_h, "p (entropy)": p_h,
                    **g.groupby(q, observed=True)["recall@100"].mean().add_prefix("recall@100 ").to_dict()})
summary = pd.DataFrame(summary).set_index("relation").T
summary.to_csv(os.path.join(OUT_DIR, "summary.csv"))
summary.round(4)